In [ ]:
from google.colab import files
files.upload()  # Upload your kaggle.json here


Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"apraveenkumarreddy","key":"1ab93157a4fb3e18dd95dabec1493bd3"}'}

In [ ]:
import os
os.environ['KAGGLE_CONFIG_DIR'] = '/content/'  # Path where kaggle.json is located


In [ ]:
!kaggle datasets list


ref                                                                title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-----------------------------------------------------------------  --------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
yashdevladdha/uber-ride-analytics-dashboard                        Uber Data Analytics Dashboard                         17324552  2025-08-08 11:13:42.920000           8905        197  1.0              
rohitgrewal/airlines-flights-data                                  Airlines Flights Data                                  2440299  2025-07-29 09:16:00.463000          19477        370  1.0              
wasiqaliyasir/breast-cancer-dataset                                Breast cancer dataset                                    49830  2025-07-30 12:52:44.057000          10753        350  1.0

In [ ]:
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset
!unzip brain-tumor-mri-dataset.zip -d data/brain_tumor_mri


Dataset URL: https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset
License(s): CC0-1.0
 94% 140M/149M [00:00<00:00, 273MB/s] 
100% 149M/149M [00:00<00:00, 320MB/s]
Archive:  brain-tumor-mri-dataset.zip
checkdir:  cannot create extraction directory: data/brain_tumor_mri
           No such file or directory


In [ ]:
import os
import zipfile

# Create directory
os.makedirs('data/brain_tumor_mri', exist_ok=True)

# Unzip the dataset into that directory
with zipfile.ZipFile('brain-tumor-mri-dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('data/brain_tumor_mri')


In [ ]:
for folder in os.listdir('data/brain_tumor_mri'):
    print(folder, ":", len(os.listdir(os.path.join('data/brain_tumor_mri', folder))))


Testing : 4
Training : 4


In [ ]:
import os

for folder in os.listdir('data/brain_tumor_mri'):
    path = os.path.join('data/brain_tumor_mri', folder)
    if os.path.isdir(path):
        print(folder, ":", len(os.listdir(path)))


Testing : 4
Training : 4


In [ ]:
import os

train_dir = 'data/brain_tumor_mri/Training'
test_dir = 'data/brain_tumor_mri/Testing'

for folder in os.listdir(train_dir):
    path = os.path.join(train_dir, folder)
    print(f"Training - {folder}:", len(os.listdir(path)))

for folder in os.listdir(test_dir):
    path = os.path.join(test_dir, folder)
    print(f"Testing - {folder}:", len(os.listdir(path)))


Training - notumor: 1595
Training - meningioma: 1339
Training - glioma: 1321
Training - pituitary: 1457
Testing - notumor: 405
Testing - meningioma: 306
Testing - glioma: 300
Testing - pituitary: 300


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import os


In [ ]:
img_size = (224, 224)
batch_size = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    'data/brain_tumor_mri/Training',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    'data/brain_tumor_mri/Testing',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)


Found 5712 images belonging to 4 classes.
Found 1311 images belonging to 4 classes.


In [ ]:
!pip install torch torchvision transformers

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os
import torch.nn as nn
import torch.optim as optim
from transformers import SwinForImageClassification, AutoImageProcessor


In [ ]:
# Image transformations for training
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

class BrainTumorDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.images = []
        self.labels = []
        self.class_names = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.class_names)}

        for cls_name in self.class_names:
            cls_folder = os.path.join(root_dir, cls_name)
            for img_name in os.listdir(cls_folder):
                self.images.append(os.path.join(cls_folder, img_name))
                self.labels.append(self.class_to_idx[cls_name])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = Image.open(self.images[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = self.labels[idx]
        return image, label

# Create datasets
train_dataset = BrainTumorDataset('data/brain_tumor_mri/Training', transform=transform)
test_dataset = BrainTumorDataset('data/brain_tumor_mri/Testing', transform=transform)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)


In [ ]:
from transformers import SwinForImageClassification

num_classes = 4

model = SwinForImageClassification.from_pretrained(
    "microsoft/swin-base-patch4-window7-224-in22k",
    num_labels=num_classes,
    ignore_mismatched_sizes=True  # <-- ignore the head size mismatch
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


Some weights of SwinForImageClassification were not initialized from the model checkpoint at microsoft/swin-base-patch4-window7-224-in22k and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([21841]) in the checkpoint and torch.Size([4]) in the model instantiated
- classifier.weight: found shape torch.Size([21841, 1024]) in the checkpoint and torch.Size([4, 1024]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


SwinForImageClassification(
  (swin): SwinModel(
    (embeddings): SwinEmbeddings(
      (patch_embeddings): SwinPatchEmbeddings(
        (projection): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
      )
      (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): SwinEncoder(
      (layers): ModuleList(
        (0): SwinStage(
          (blocks): ModuleList(
            (0): SwinLayer(
              (layernorm_before): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
              (attention): SwinAttention(
                (self): SwinSelfAttention(
                  (query): Linear(in_features=128, out_features=128, bias=True)
                  (key): Linear(in_features=128, out_features=128, bias=True)
                  (value): Linear(in_features=128, out_features=128, bias=True)
                  (dropout): Dropout(p=0.0, inplace=False)
                )
                (output): SwinSelfOutput(

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=3e-5)


In [ ]:
epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images).logits
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {running_loss/len(train_loader):.4f} | Accuracy: {100*correct/total:.2f}%")


Epoch 1/5 | Loss: 0.2181 | Accuracy: 91.82%
Epoch 2/5 | Loss: 0.0480 | Accuracy: 98.32%
Epoch 3/5 | Loss: 0.0176 | Accuracy: 99.51%
Epoch 4/5 | Loss: 0.0193 | Accuracy: 99.42%
Epoch 5/5 | Loss: 0.0121 | Accuracy: 99.61%


In [ ]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images).logits
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100*correct/total:.2f}%")


Test Accuracy: 95.19%
